# DAX schema exploration — replacing TMDL parsing

Scrap notebook, not committed app code. Goal: pull everything
`scripts/build_model_context.py` currently gets from `.pbip` TMDL parsing,
using live Power BI APIs instead.

**Confirmed split, two APIs:**
- `executeQueries` (`INFO.VIEW.*` DAX functions) — tables, columns,
  measure names, relationships. No Premium needed, works on Contributor.
- Scanner API (`admin/workspaces/getInfo` + `datasetExpressions=true`) —
  the actual DAX `Expression` text, which `executeQueries` returns as
  `null` regardless of workspace role (confirmed up to Admin). Needed:
  three tenant settings ("Allow service principals to use read-only admin
  APIs", "...detailed metadata", "...DAX and mashup expressions"), all
  scoped to the same security group already used for `executeQueries`.

Also confirmed: what-if/field parameter tables are identifiable from the
Scanner API's measure expressions — their auto-generated value measures are
exactly `SELECTEDVALUE('Table'[Column], default)`, naming the parameter
table and column directly.

In [ ]:
import json
import re
import sys
import time

import msal
import requests

sys.path.insert(0, "..")  # so `app.*` resolves from notebooks/'s cwd

from app.config import GCP_PROJECT_ID, get_secret

DATASET_ID = "03fe95af-12af-4e00-bbd6-241e942f76bc"
WORKSPACE_ID = "8e20abd3-703e-46b8-9832-950249767864"

In [ ]:
app = msal.ConfidentialClientApplication(
    get_secret("power-bi-sp-client-id", GCP_PROJECT_ID),
    authority=f"https://login.microsoftonline.com/{get_secret('azure-tenant-id', GCP_PROJECT_ID)}",
    client_credential=get_secret("power-bi-sp-client-secret", GCP_PROJECT_ID),
)
result = app.acquire_token_for_client(
    scopes=["https://analysis.windows.net/powerbi/api/.default"]
)
assert "access_token" in result, result
token = result["access_token"]
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
print("token acquired")

In [46]:
def run_dax(query: str) -> list[dict]:
    resp = requests.post(
        f"https://api.powerbi.com/v1.0/myorg/datasets/{DATASET_ID}/executeQueries",
        headers=headers,
        json={"queries": [{"query": query}], "serializerSettings": {"includeNulls": True}},
    )
    resp.raise_for_status()
    return resp.json()["results"][0]["tables"][0]["rows"]

## Tables

In [53]:
tables = run_dax(
    'EVALUATE SELECTCOLUMNS(INFO.VIEW.TABLES(), '
    '"TableName", [Name], "Description", [Description], '
    '"IsHidden", [IsHidden])'
)
print(len(tables), "tables")
tables[:5]

25 tables


[{'[TableName]': 'beeswarm_plot_data',
  '[Description]': 'Data for beeswarm plot',
  '[IsHidden]': False},
 {'[TableName]': 'dataset_split_dimension',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': 'evaluation_metrics',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': 'model_types_dimension',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': 'models_dimension',
  '[Description]': None,
  '[IsHidden]': False}]

## Columns

In [54]:
columns = run_dax(
    'EVALUATE SELECTCOLUMNS(INFO.VIEW.COLUMNS(), '
    '"TableName", [Table], "ColumnName", [Name], "DataType", [DataType], '
    '"Description", [Description], "IsHidden", [IsHidden])'
)
print(len(columns), "columns")
columns[:10]

106 columns


[{'[TableName]': 'beeswarm_plot_data',
  '[ColumnName]': 'RowNumber-2662979B-1795-4F74-8F37-6A1BA8059B61',
  '[DataType]': 'Integer',
  '[Description]': None,
  '[IsHidden]': True},
 {'[TableName]': 'beeswarm_plot_data',
  '[ColumnName]': 'unique_plot_id',
  '[DataType]': 'Integer',
  '[Description]': 'unique plot id for beeswarm chart',
  '[IsHidden]': False},
 {'[TableName]': 'beeswarm_plot_data',
  '[ColumnName]': 'user_id',
  '[DataType]': 'Integer',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': 'beeswarm_plot_data',
  '[ColumnName]': 'anchor_order_number',
  '[DataType]': 'Integer',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': 'beeswarm_plot_data',
  '[ColumnName]': 'label_reordered',
  '[DataType]': 'Integer',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': 'beeswarm_plot_data',
  '[ColumnName]': 'prediction',
  '[DataType]': 'Number',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': 'beeswarm_plot_data',

## Relationships

In [49]:
relationships = run_dax(
    'EVALUATE SELECTCOLUMNS(INFO.VIEW.RELATIONSHIPS(), '
    '"FromTable", [FromTable], "FromColumn", [FromColumn], '
    '"ToTable", [ToTable], "ToColumn", [ToColumn], '
    '"FromCardinality", [FromCardinality], "ToCardinality", [ToCardinality], '
    '"CrossFilteringBehavior", [CrossFilteringBehavior], "IsActive", [IsActive])'
)
print(len(relationships), "relationships")
relationships[:5]

9 relationships


[{'[FromTable]': 'beeswarm_plot_data',
  '[FromColumn]': 'model_type',
  '[ToTable]': 'model_types_dimension',
  '[ToColumn]': 'model_type',
  '[FromCardinality]': 'Many',
  '[ToCardinality]': 'One',
  '[CrossFilteringBehavior]': 'OneDirection',
  '[IsActive]': True},
 {'[FromTable]': 'beeswarm_plot_data',
  '[FromColumn]': 'split',
  '[ToTable]': 'dataset_split_dimension',
  '[ToColumn]': 'dataset_split',
  '[FromCardinality]': 'Many',
  '[ToCardinality]': 'One',
  '[CrossFilteringBehavior]': 'OneDirection',
  '[IsActive]': True},
 {'[FromTable]': 'beeswarm_plot_data',
  '[FromColumn]': 'model_name',
  '[ToTable]': 'base_models_dimension',
  '[ToColumn]': 'model_name',
  '[FromCardinality]': 'Many',
  '[ToCardinality]': 'One',
  '[CrossFilteringBehavior]': 'OneDirection',
  '[IsActive]': True},
 {'[FromTable]': 'evaluation_metrics',
  '[FromColumn]': 'Model',
  '[ToTable]': 'models_dimension',
  '[ToColumn]': 'Model',
  '[FromCardinality]': 'Many',
  '[ToCardinality]': 'One',
  '[Cros

## Measures — names only via `executeQueries`

`Expression`/`FormatString` are `null` here regardless of role. Names,
table, and `IsHidden` are fine — full data comes from the Scanner API below.

In [ ]:
measures_names = run_dax(
    'EVALUATE SELECTCOLUMNS(INFO.VIEW.MEASURES(), '
    '"TableName", [Table], "MeasureName", [Name], "Description", [Description], '
    '"IsHidden", [IsHidden])'
)
print(len(measures_names), "measures")
measures_names[:5]

69 measures


[{'[TableName]': '_Model_Evaluation_Measures',
  '[MeasureName]': 'Recall',
  '[Description]': 'Model Evaluation metric: Recall at 5',
  '[IsHidden]': False},
 {'[TableName]': '_Model_Evaluation_Measures',
  '[MeasureName]': 'Precision',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': '_Model_Evaluation_Measures',
  '[MeasureName]': 'NDCG',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': '_Model_Evaluation_Measures',
  '[MeasureName]': 'F1',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': '_Model_Evaluation_Measures',
  '[MeasureName]': 'Dynamic_Metric_Description',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': '_Model_Evaluation_Measures',
  '[MeasureName]': 'Champion Recall',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': '_Model_Evaluation_Measures',
  '[MeasureName]': 'Champion Precision',
  '[Description]': None,
  '[IsHidden]': False},
 {'[TableName]': '_Model_Evaluation_Measures',
  '[Me

## Measures — full data (Description, Expression, FormatString) via Scanner API

In [55]:
def scan_workspace(workspace_id: str) -> dict:
    submit = requests.post(
        "https://api.powerbi.com/v1.0/myorg/admin/workspaces/getInfo",
        headers=headers,
        params={"datasetExpressions": "True", "datasetSchema": "True", "lineage": "True"},
        json={"workspaces": [workspace_id]},
    )
    submit.raise_for_status()
    scan_id = submit.json()["id"]

    for _ in range(20):
        status = requests.get(
            f"https://api.powerbi.com/v1.0/myorg/admin/workspaces/scanStatus/{scan_id}",
            headers=headers,
        ).json()["status"]
        if status == "Succeeded":
            break
        time.sleep(2)
    else:
        raise TimeoutError("Scan did not finish in time")

    return requests.get(
        f"https://api.powerbi.com/v1.0/myorg/admin/workspaces/scanResult/{scan_id}",
        headers=headers,
    ).json()


scan_result = scan_workspace(WORKSPACE_ID)
dataset = next(
    ds
    for ws in scan_result["workspaces"]
    for ds in ws.get("datasets", [])
    if ds["id"] == DATASET_ID
)
print("scan ok,", len(dataset["tables"]), "tables in scan result")

scan ok, 23 tables in scan result


In [56]:
measures_full = [
    {
        "table": tbl["name"],
        "name": m["name"],
        "description": m.get("description"),
        "expression": m.get("expression"),
        "formatString": m.get("formatString"),
        "isHidden": m.get("isHidden"),
    }
    for tbl in dataset["tables"]
    for m in tbl.get("measures", [])
]
print(len(measures_full), "measures with full data")
with_expression = sum(1 for m in measures_full if m["expression"])
print(f"{with_expression}/{len(measures_full)} have a non-null Expression")
measures_full[:5]

69 measures with full data
69/69 have a non-null Expression


[{'table': '_Model_Evaluation_Measures',
  'name': 'Recall',
  'description': 'Model Evaluation metric: Recall at 5',
  'expression': 'SUM(evaluation_metrics[Recall_at_5])',
  'formatString': None,
  'isHidden': False},
 {'table': '_Model_Evaluation_Measures',
  'name': 'Precision',
  'description': None,
  'expression': 'SUM(evaluation_metrics[Precision_at_5])',
  'formatString': None,
  'isHidden': False},
 {'table': '_Model_Evaluation_Measures',
  'name': 'NDCG',
  'description': None,
  'expression': 'SUM(evaluation_metrics[NDCG_at_5])',
  'formatString': None,
  'isHidden': False},
 {'table': '_Model_Evaluation_Measures',
  'name': 'F1',
  'description': None,
  'expression': 'SUM(evaluation_metrics[F1_at_5])',
  'formatString': None,
  'isHidden': False},
 {'table': '_Model_Evaluation_Measures',
  'name': 'Dynamic_Metric_Description',
  'description': None,
  'expression': '\nVAR SelectedMetricText = SELECTEDVALUE(\'Evaluation Metric Parameter\'[Parameter Fields])\nVAR Descriptio

## Parameter detection — `SELECTEDVALUE('Table'[Column], default)`

Two passes: a strict one (the measure's *entire* expression is a bare
`SELECTEDVALUE(...)` call — the classic auto-generated what-if parameter
value measure), and a broad one (the pattern appears *anywhere* in the
expression — catches parameter usage inside larger measures, like a field
parameter driving a `SWITCH`). The broad pass doesn't mean that measure
*is* the parameter's value measure, just that it reads one.

In [ ]:
SELECTEDVALUE_RE = re.compile(r"SELECTEDVALUE\(\s*'([^']+)'\[([^\]]+)\](?:\s*,\s*(.+?))?\s*\)")


def find_selectedvalue_refs(expr: str | None) -> list[tuple[str, str, str | None]]:
    if not expr:
        return []
    return [(t, c, d.strip() if d else None) for t, c, d in SELECTEDVALUE_RE.findall(expr)]


parameter_value_measures = []
parameter_references = []
for m in measures_full:
    expr = (m["expression"] or "").strip()
    refs = find_selectedvalue_refs(expr)
    for table, column, default in refs:
        parameter_references.append({"measure": m["name"], "table": table, "column": column})
    if refs and expr.startswith("SELECTEDVALUE("):
        table, column, default = refs[0]
        parameter_value_measures.append(
            {"value_measure": m["name"], "table": table, "column": column, "default": default}
        )

print(len(parameter_value_measures), "clean value measures")
print(parameter_value_measures)
print()
print(len(parameter_references), "total SELECTEDVALUE references (superset)")
print(parameter_references)

## Calculated / non-Import tables

Checking whether any table is a calculated table (field parameters and old
what-if range tables are both calculated tables under the hood) and what
fields the Scanner API exposes for them — `executeQueries`'s
`INFO.VIEW.TABLES()` returned zero calculated tables, which may mean there
are none, or may mean `Expression` was gated there too. Scanner API data
isn't gated, so this is the real answer.

In [ ]:
non_import_tables = [
    t for t in dataset["tables"] if t.get("storageMode") != "Import"
]
print(len(non_import_tables), "non-Import tables")
non_import_tables